# Amazon Fine Food Reviews Data Analysis

## Context of Dataset

This dataset consists of reviews of fine foods from amazon. The data span a period of more than 10 years. Reviews include product and user information, ratings, and a plain text review. It also includes reviews from all other Amazon categories.

## Project Goal

The goal of this project is to explore the Amazon Fine Food Reviews dataset using Pandas and Polars. The analysis includes inspecting the dataset, checking data quality, filtering and grouping reviews, creating a visualization, and beginning an exploration of a machine learning model.

## The machine learning task will use review text to predict whether a review is positive or negative.

## 1.Load Packages

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import polars as pl
import seaborn as sns

## 2. Import the Dataset and Inspect the Data

The dataset is loaded from the local `data` directory using Pandas, since the CSV itself is too large to upload to Github (300 Mb). The loading time is recorded so it can later be compared with Polars.

In [2]:
DATA_PATH = Path("data") / "Reviews.csv"

reviews = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {reviews.shape}")

Dataset shape: (568454, 10)


In [3]:
reviews.head(5)

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


**Finding**: Here we can see that the original `Time` column contains Unix timestamps, and we want these values to be converted into readable dates. We will do the date conversion.

In [10]:
reviews["ReviewDate"] = pd.to_datetime(reviews["Time"], unit="s")

reviews[["Time", "ReviewDate"]].head()

,Time,ReviewDate
0,1303862400,2011-04-27
1,1346976000,2012-09-07
2,1219017600,2008-08-18
3,1307923200,2011-06-13
4,1350777600,2012-10-21


In [4]:
reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 568454 entries, 0 to 568453
Data columns (total 10 columns):
 #   Column                  Non-Null Count   Dtype
---  ------                  --------------   -----
 0   Id                      568454 non-null  int64
 1   ProductId               568454 non-null  str  
 2   UserId                  568454 non-null  str  
 3   ProfileName             568428 non-null  str  
 4   HelpfulnessNumerator    568454 non-null  int64
 5   HelpfulnessDenominator  568454 non-null  int64
 6   Score                   568454 non-null  int64
 7   Time                    568454 non-null  int64
 8   Summary                 568427 non-null  str  
 9   Text                    568454 non-null  str  
dtypes: int64(5), str(5)
memory usage: 43.4 MB


In [11]:
print(f"Earliest review date: {reviews['ReviewDate'].min()}")
print(f"Latest review date: {reviews['ReviewDate'].max()}")

Earliest review date: 1999-10-08 00:00:00
Latest review date: 2012-10-26 00:00:00


**Finding:** After converting the Unix timestamps, the dataset was found to cover reviews from 1999-10-08 through 2012-10-26.

### Summary Statistics

Descriptive statistics are calculated for the numerical columns to examine their distributions, ranges, and average values.

In [5]:
reviews.describe()

,Id,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time
count,568454.000000,568454.000000,568454.00000,568454.000000,5.684540e+05
mean,284227.500000,1.743817,2.22881,4.183199,1.296257e+09
std,164098.679298,7.636513,8.28974,1.310436,4.804331e+07
min,1.000000,0.000000,0.00000,1.000000,9.393408e+08
25%,142114.250000,0.000000,0.00000,4.000000,1.271290e+09
50%,284227.500000,0.000000,1.00000,5.000000,1.311120e+09
75%,426340.750000,2.000000,2.00000,5.000000,1.332720e+09
max,568454.000000,866.000000,923.00000,5.000000,1.351210e+09


### Check for Missing Values or Duplicated Rows


In [9]:
missing_values = reviews.isna().sum()

missing_values

Id                         0
ProductId                  0
UserId                     0
ProfileName               26
HelpfulnessNumerator       0
HelpfulnessDenominator     0
Score                      0
Time                       0
Summary                   27
Text                       0
dtype: int64

**Finding:** The dataset contains 26 missing values in `ProfileName` and 27 missing values in `Summary`. All other columns have no missing values. Because the missing values represent a very small proportion of the dataset and do not affect the variables used for the planned machine learning analysis, no rows are removed.

In [ ]:
duplicate_count = reviews.duplicated().sum()

print(f"Number of duplicate rows: {duplicate_count}")

Number of exact duplicate rows: 0


In [8]:
print(f"Number of unique IDs: {reviews['Id'].nunique()}")
print(f"Total number of rows: {len(reviews)}")

Number of unique IDs: 568454
Total number of rows: 568454


**Finding**: There are no duplicated rows, so we will continue with the filtering and grouping in Pandas.